# B3LZ — Local zoom inside merged regions (on the LM0 base)

## Why
- **LM0** (SAM2.1-Large + masked input M0) gives the best sparse and medium scenes (H1b-K3n[LM0]: S1 6.92, S2 6.24) but loses dense objects.
- **Zoom** (B3Z) gives by far the best dense scenes (S6 down to 209–243) by magnifying objects 2×, but applied to the whole image it adds part-masks in sparse and medium scenes that masking cannot remove (Z-LM0: S1 10.5–14.0, S3 15–17).
- **Local zoom** applies zoom only where SAM2 merged objects — inside rule-A merged regions — so sparse and medium images (few such regions) keep LM0's profile, and dense merged masks get zoom's splitting. Held-out cost: ~1–2 h on top of LM0 (vs ~13–15 h for full Z-LM0).

## Pre-declared design (before any B3LZ result)
| Choice | Rule |
|---|---|
| Base | LM0 GPV_R4 mask set (B3L §5 cache: SAM2.1-Large, masked input M0) |
| Regions | rule A on that set; region = flagged mask − unflagged kept masks − regions of higher-scoring flagged masks (SNP-v2 / GPV_K definition) |
| Zoom crop | region bounding box padded by 10% of its width/height per side (≥ 1 px), clipped to the image; cut from the **masked** image; SAM2 resizes it to its full input |
| **(a) Zoomed prompts** | each cached `dense_recovery` prompt inside the region, predicted on the crop with SAM2.1-Large (one positive point, multimask off) |
| **(b) Zoomed automatic generation** | SAM2.1-Large automatic mask generator on the crop (`points_per_side` 32, no further crops, `apply_postprocessing=False`) |
| Keep / replace | a new mask is kept if > 50% of it lies inside the region; a flagged mask is replaced if it gets ≥ 1 new mask (otherwise kept); then IoU-NMS 0.5 over the whole set |
| Counting | **GPV_R4**, **H1b**, **GPV_K-n**, **H1b-K3n** (B3MN definitions; normals edge map from the B3MN cache) on the new mask set |
| Targets | T1 (S1, S2 vs B0_v1), T2′ (S5, S6 vs H1b_v1), T3′ (reweighted MAE vs H1b_v1) |

## Run order
T4 → 1.0–1.3 → 2.1–2.2 → 3.1 (resumable) → 4.1.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3LZ_Local_Zoom"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3LZ_Local_Zoom


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


In [4]:
# ============================================================
# Step 1.3 — Normals edge map and K3n counting (B3MN definitions)
# ============================================================

from scipy import ndimage
import src.pipeline.multiplicity as K
importlib.reload(K)
N_ROOT = TD_ROOT / "normals"

def normal_edge_map(nrm):
    n = nrm.astype(np.float32)
    n = n / np.maximum(np.linalg.norm(n, axis=2, keepdims=True), 1e-6)
    edge = np.zeros(n.shape[:2], np.float32)
    for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        edge = np.maximum(edge, 1.0 - (n * np.roll(n, shift=(dy, dx), axis=(0, 1))).sum(axis=2))
    edge[0, :] = edge[1, :]; edge[-1, :] = edge[-2, :]; edge[:, 0] = edge[:, 1]; edge[:, -1] = edge[:, -2]
    return ndimage.gaussian_filter(edge, 1.0)

def k3n_segments(region, edge, min_seg, h_frac=0.1):
    e = edge[region]
    h = h_frac * float(np.percentile(e, 95) - np.percentile(e, 5))
    return 1 if h <= 0 else K._marker_watershed(edge, region, -edge, h, min_seg)

def gpv_k_n_count(r4_m, r4_s, d1, edge):
    masks = np.asarray(r4_m, bool)
    if masks.shape[0] == 0:
        return 0
    flagged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, r4_s, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0])
    return n_nonflag + sum(max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions)

def h1b_k3n_count(r4_m, d1, a3c, depth, edge):
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = r4_m[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            total += max(1, k3n_segments(rg, edge, 0.25 * a_ref))
    return float(total)

def all_counts(masks, scores, d1, a3c, depth, edge):
    return {"GPV_R4": int(masks.shape[0]),
            "H1b": float(hybrid.h1b_count(masks, d1, a3c, depth, SG_CFG)[0]),
            "GPV_K-n": gpv_k_n_count(masks, scores, d1, edge),
            "H1b-K3n": h1b_k3n_count(masks, d1, a3c, depth, edge)}
print("K3n counting ready")

K3n counting ready


# 2. SAM2.1-Large

In [5]:
# ============================================================
# Step 2.1 — Install SAM2 (frozen commit)
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
print("SAM2 commit:", subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.8 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [6]:
# ============================================================
# Step 2.2 — SAM2.1-Large predictor and automatic mask generator
# ============================================================

SAM2_L = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
CFG_L = "configs/sam2.1/sam2.1_hiera_l.yaml"
assert SAM2_L.exists(), "Run B3L Step 2.2 once to download SAM2.1-Large."
predictor_L = SAM2ImagePredictor(build_sam2(CFG_L, str(SAM2_L), device=DEVICE).eval())
amg_L = b0.build_amg(SAM2_L, device=DEVICE, model_cfg=CFG_L, verify_sha256=False)   # points_per_side 32, no crops
print("SAM2.1-Large predictor and AMG ready")

SAM2.1-Large predictor and AMG ready


# 3. Local zoom on the LM0 base

In [7]:
# ============================================================
# Step 3.1 — Local zoom, options (a) prompts and (b) automatic generation (resumable)
# ============================================================

TD_TILED = TD_ROOT / "san_vocab_tiled"
LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
ROWS = OUT_DIR / "rows"
ROWS.mkdir(parents=True, exist_ok=True)
PAD_FRAC = 0.10

def safe_amg(image):
    try:
        m, meta = b0.run_amg(amg_L, image)
        return m, np.asarray(meta["predicted_iou"], float)
    except IndexError:                    # SAM2 empty-result case
        return np.zeros((0,) + image.shape[:2], bool), np.zeros(0)

def local_zoom(r4_m, r4_s, prompts, d1, image, option):
    n = r4_m.shape[0]
    diag = {"flagged": 0, "regions_zoomed": 0, "replaced": 0, "new_masks": 0}
    if n == 0:
        return r4_m, np.asarray(r4_s, float), diag
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    diag["flagged"] = int(merged.sum())
    if not merged.any():
        return r4_m, np.asarray(r4_s, float), diag
    H, W = r4_m.shape[1:]
    taken = r4_m[~merged].any(axis=0) if (~merged).any() else np.zeros((H, W), bool)
    if len(prompts):
        px = np.clip(np.rint(prompts[:, 0]).astype(int), 0, W - 1)
        py = np.clip(np.rint(prompts[:, 1]).astype(int), 0, H - 1)
    replaced, new_m, new_s = set(), [], []
    for i in [j for j in np.argsort(-np.asarray(r4_s), kind="stable") if merged[j]]:
        region = r4_m[i] & ~taken
        taken |= region
        if not region.any():
            continue
        ys, xs = np.nonzero(region)
        bh, bw = ys.max() - ys.min() + 1, xs.max() - xs.min() + 1
        py_, px_ = max(1, int(round(PAD_FRAC * bh))), max(1, int(round(PAD_FRAC * bw)))
        y0, y1 = max(0, ys.min() - py_), min(H, ys.max() + 1 + py_)
        x0, x1 = max(0, xs.min() - px_), min(W, xs.max() + 1 + px_)
        crop = np.ascontiguousarray(image[y0:y1, x0:x1])
        diag["regions_zoomed"] += 1
        got = []
        if option == "a":
            inside = np.flatnonzero(region[py, px]) if len(prompts) else np.zeros(0, int)
            if len(inside):
                with torch.inference_mode():
                    predictor_L.set_image(crop)
                    for k in inside:
                        m, s, _ = predictor_L.predict(point_coords=np.asarray([[prompts[k][0] - x0, prompts[k][1] - y0]], np.float32),
                                                      point_labels=np.asarray([1], np.int32), multimask_output=False)
                        full = np.zeros((H, W), bool); full[y0:y1, x0:x1] = np.asarray(m[0], bool)
                        got.append((full, float(np.asarray(s)[0])))
        else:
            cm, cs = safe_amg(crop)
            for m, sc in zip(cm, cs):
                full = np.zeros((H, W), bool); full[y0:y1, x0:x1] = m
                got.append((full, float(sc)))
        got = [(m, s) for m, s in got if m.sum() and (m & region).sum() > 0.5 * m.sum()]
        if got:
            replaced.add(i)
            new_m += [g[0] for g in got]; new_s += [g[1] for g in got]
    diag["replaced"], diag["new_masks"] = len(replaced), len(new_m)
    keep_idx = [j for j in range(n) if j not in replaced]
    pool = np.concatenate([r4_m[keep_idx], np.asarray(new_m, bool).reshape(-1, H, W)])
    scores = np.concatenate([np.asarray(r4_s, float)[keep_idx], np.asarray(new_s, float)])
    kk = gpv.nms_masks(pool, scores, 0.5)
    return pool[kk], scores[kk], diag

todo = [r for r in td_df.itertuples(index=False) if not (ROWS / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for i, r in enumerate(tqdm(todo, desc="Local zoom")):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    prompts = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")["prompts_xy"]
    with np.load(N_ROOT / f"{stem}.npz") as v:
        edge = normal_edge_map(v["normals"])
    with np.load(TD_TILED / f"{stem}.npz") as v:
        reg = v["tiled_argmax"] == 0
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
    masked = (img * reg[..., None]).astype(np.uint8)
    bL, bLmeta = io_cache.load_b0_masks(LM_ROOT / "b0" / f"{int(di):05d}_{stem}.npz")
    gL = io_cache.load_guided_masks(LM_ROOT / "guided" / f"{int(di):04d}_{stem}.npz")
    kgL = gpv.nms_masks(gL["masks"], gL["scores"], 0.5) if gL["masks"].shape[0] else np.zeros(0, int)
    r4_m, r4_s, _ = gpv.build_r4(gL["masks"][kgL], np.asarray(gL["scores"])[kgL], bL, bLmeta["predicted_iou"], 0.5)
    row = {"dataset_index": int(di)}
    for opt in ["a", "b"]:
        nm, ns, dg = local_zoom(r4_m, r4_s, prompts, d1, masked, opt)
        row.update({f"{k}[LZ{opt}]": v for k, v in all_counts(nm, ns, d1, a3c, dep, edge).items()})
        row.update({f"{k}[LZ{opt}]": v for k, v in dg.items()})
    with open(ROWS / f"{int(di):05d}.json", "w") as f:
        json.dump(row, f)
    if (i + 1) % 25 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Local zoom:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 121.6 min


# 4. Results and targets

In [8]:
# ============================================================
# Step 4.1 — Diagnostics, metrics and targets
# ============================================================

ROWS = OUT_DIR / "rows"
lz = pd.DataFrame([json.load(open(p)) for p in sorted(ROWS.glob("*.json"))])
assert len(lz) == 400, len(lz)
mn = pd.read_csv(RESULTS_ROOT / "B3MN_Marigold_Normals" / "section4_bases" / "B3MN_s4_per_image.csv")
ref_cols = [c for c in ["H1b[LM0]", "GPV_K-n[LM0]", "H1b-K3n[LM0]", "H1b[Z-LM0]", "GPV_K-n[Z-LM0]", "H1b-K3n[Z-LM0]"]
            if c in mn.columns]
frame = (lz.merge(mn[["dataset_index"] + ref_cols], on="dataset_index")
         .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one"))

print("Flagged masks, zoomed regions and new masks per image, by stratum:")
dcols = [f"{k}[LZ{o}]" for o in ["a", "b"] for k in ["flagged", "regions_zoomed", "replaced", "new_masks"]]
display(frame.groupby("density_stratum")[dcols].mean().reindex(STRATA).round(2))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1"] + ref_cols + \
      [f"{k}[LZ{o}]" for o in ["a", "b"] for k in ["GPV_R4", "H1b", "GPV_K-n", "H1b-K3n"]]
res, targets = evaluate(frame, SYS, "Local zoom on the LM0 base (train_dev400)")

frame.to_csv(OUT_DIR / "B3LZ_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3LZ_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3LZ_targets.csv", index=False)
with open(OUT_DIR / "B3LZ_metadata.json", "w") as f:
    json.dump({"notebook": "B3LZ_Local_Zoom",
               "base": "LM0 GPV_R4 set (SAM2.1-Large, masked input M0; B3L s5)",
               "regions": "rule A flags; region = flagged - unflagged kept masks - earlier flagged regions (desc score)",
               "crop": f"region bbox padded {PAD_FRAC:.0%} per side, from the masked image",
               "options": {"a": "region prompts predicted on the crop (SAM2.1-Large, single point, multimask off)",
                           "b": "SAM2.1-Large AMG on the crop (points_per_side 32, no crops)"},
               "keep": "new mask > 50% inside region; replace flagged mask if >= 1 new mask; NMS 0.5",
               "counting": ["GPV_R4", "H1b", "GPV_K-n", "H1b-K3n"],
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Flagged masks, zoomed regions and new masks per image, by stratum:


,flagged[LZa],regions_zoomed[LZa],replaced[LZa],new_masks[LZa],flagged[LZb],regions_zoomed[LZb],replaced[LZb],new_masks[LZb]
density_stratum,,,,,,,,
S1_0_20,0.80,0.80,0.65,5.38,0.80,0.80,0.46,1.38
S2_20_40,0.76,0.72,0.49,3.90,0.76,0.72,0.26,0.59
S3_40_60,0.92,0.92,0.63,3.43,0.92,0.92,0.32,0.82
S4_60_80,1.28,1.23,1.02,4.40,1.28,1.23,0.73,1.88
S5_80_95,1.78,1.75,1.30,6.85,1.78,1.75,0.95,4.12
S6_95_100,2.00,1.75,1.30,18.42,2.00,1.75,1.17,8.63


Local zoom on the LM0 base (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,H1b[LM0],28.91,-49.22,6.89,6.14,9.93,14.90,42.42,299.43
4,GPV_K-n[LM0],27.71,-41.95,8.31,5.62,10.73,15.57,39.80,273.78
5,H1b-K3n[LM0],28.24,-44.33,6.92,6.24,11.62,15.55,38.97,286.55
6,H1b[Z-LM0],27.28,-33.92,10.54,8.95,15.90,16.12,32.32,242.67
7,GPV_K-n[Z-LM0],27.02,-30.56,13.98,9.43,15.35,16.43,30.90,227.00
8,H1b-K3n[Z-LM0],27.25,-28.88,10.66,8.90,17.23,16.67,30.67,239.22
9,GPV_R4[LZa],30.18,-53.26,5.74,5.44,10.17,15.80,46.72,314.80


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,H1b[LM0],"+1.33 [-0.44, +3.11]","-1.60 [-3.34, +0.16]","+5.08 [-0.08, +11.52]","+24.45 [+12.60, +38.02]","+0.26 [-1.03, +1.64]",True,False,False,False
1,GPV_K-n[LM0],"+2.75 [+0.25, +5.49]","-2.11 [-3.86, -0.24]","+2.47 [-6.15, +12.07]","-1.20 [-28.62, +23.70]","-0.94 [-3.05, +1.25]",False,True,False,False
2,H1b-K3n[LM0],"+1.36 [-0.39, +3.10]","-1.50 [-3.26, +0.29]","+1.63 [-5.00, +8.97]","+11.57 [-9.08, +31.38]","-0.41 [-2.10, +1.39]",True,True,False,False
3,H1b[Z-LM0],"+4.97 [+2.62, +7.50]","+1.21 [-1.18, +3.80]","-5.02 [-10.83, +2.20]","-32.32 [-51.35, -12.13]","-1.37 [-3.03, +0.38]",False,True,False,False
4,GPV_K-n[Z-LM0],"+8.41 [+4.65, +12.46]","+1.69 [-0.83, +4.45]","-6.43 [-14.73, +2.92]","-47.98 [-76.67, -21.10]","-1.63 [-3.91, +0.73]",False,True,False,False
5,H1b-K3n[Z-LM0],"+5.10 [+2.62, +7.78]","+1.16 [-1.26, +3.78]","-6.67 [-13.27, +0.98]","-35.77 [-56.47, -14.38]","-1.40 [-3.25, +0.58]",False,True,False,False
6,GPV_R4[LZa],"+0.17 [-1.34, +1.71]","-2.30 [-3.85, -0.66]","+9.38 [+2.85, +17.65]","+39.82 [+25.98, +54.83]","+1.53 [-0.00, +3.16]",True,False,False,False
7,H1b[LZa],"+1.50 [-0.34, +3.44]","-1.54 [-3.31, +0.26]","+4.90 [-0.23, +11.38]","+22.47 [+10.35, +36.12]","+0.20 [-1.08, +1.58]",True,False,False,False
8,GPV_K-n[LZa],"+3.19 [+0.46, +6.25]","-1.77 [-3.66, +0.29]","+3.57 [-4.75, +12.88]","-5.32 [-34.53, +21.65]","-0.88 [-3.06, +1.29]",False,True,False,False
9,H1b-K3n[LZa],"+1.39 [-0.41, +3.24]","-1.41 [-3.20, +0.42]","+2.22 [-4.12, +9.32]","+9.28 [-11.25, +29.18]","-0.44 [-2.09, +1.30]",True,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3LZ_Local_Zoom
